# 02 — Integração dos Dois Períodos e Limpeza dos Dados

Integração das bases do INPE (BDQueimadas, satélite `NOAA-20`) referentes aos ciclos 2024–2025 (GeoJSON) e 2025–2026 (CSV), classificação dos atributos, padronização, tratamento de inconsistências/sentinelas, imputação de nulos e exportação da base limpa em Parquet.

## 1. Bibliotecas e configuração de ambiente (Local / Google Colab)

In [1]:
import importlib.util
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    repo_dados = Path("/content/PROJETO-COLETA-E-RASPAGEM-DE-DADOS/dados")
    if repo_dados.exists():
        DATA_DIR = repo_dados
    else:
        from google.colab import drive

        drive.mount("/content/drive")
        DATA_DIR = Path("/content/drive/MyDrive/PROJETO-COLETA-E-RASPAGEM-DE-DADOS/dados")
else:
    DATA_DIR = Path("../dados")

ARQUIVO_GEOJSON = DATA_DIR / "bdqueimadas_2024-11-30_2025-11-30.geojson"
ARQUIVO_CSV = DATA_DIR / "bdqueimadas_2025-12-01_2026-11-19.csv"
ARQUIVO_PARQUET = DATA_DIR / "queimadas_limpo.parquet"

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"GeoJSON encontrado: {ARQUIVO_GEOJSON.exists()} ({ARQUIVO_GEOJSON})")
print(f"CSV encontrado:     {ARQUIVO_CSV.exists()} ({ARQUIVO_CSV})")

Ambiente: Local
GeoJSON encontrado: True (..\dados\bdqueimadas_2024-11-30_2025-11-30.geojson)
CSV encontrado:     True (..\dados\bdqueimadas_2025-12-01_2026-11-19.csv)


## 2. Leitura e integração das duas bases

A coluna `geometry` do GeoJSON é descartada antes da concatenação porque `Latitude` e `Longitude` já estão presentes em ambas as bases.

In [2]:
df_geojson = gpd.read_file(ARQUIVO_GEOJSON).drop(columns=["geometry"])
df_csv = pd.read_csv(ARQUIVO_CSV, encoding="utf-8-sig")

df_geojson["origem_formato"] = "GeoJSON (2024-2025)"
df_csv["origem_formato"] = "CSV (2025-2026)"

df = pd.concat([df_geojson, df_csv], ignore_index=True)

print(f"GeoJSON: {len(df_geojson):,} linhas")
print(f"CSV:     {len(df_csv):,} linhas")
print(f"Total:   {len(df):,} linhas x {df.shape[1]} colunas")

GeoJSON: 858,405 linhas
CSV:     780,217 linhas
Total:   1,638,622 linhas x 13 colunas


## 3. Validação da integração

In [3]:
assert len(df) == 1_638_622, f"Esperado 1.638.622 linhas, obtido {len(df)}"

datas_geojson = set(pd.to_datetime(df_geojson["DataHora"]).dt.date)
datas_csv = set(
    pd.to_datetime(
        df_csv["DataHora"].astype(str).str.replace("\ufeff", "", regex=False),
        format="mixed",
    ).dt.date
)
dias_em_comum = datas_geojson & datas_csv
assert len(dias_em_comum) == 0, f"Sobreposição encontrada: {sorted(dias_em_comum)}"

print(f"Total confirmado: {len(df):,} linhas")
print(f"Intervalo GeoJSON: {min(datas_geojson)} a {max(datas_geojson)}")
print(f"Intervalo CSV:     {min(datas_csv)} a {max(datas_csv)}")
print(f"Dias em comum entre os arquivos: {len(dias_em_comum)}")

nulos = pd.DataFrame(
    {
        "nulos": df.isna().sum(),
        "percentual (%)": (df.isna().mean() * 100).round(4),
    }
)
display(nulos)

Total confirmado: 1,638,622 linhas
Intervalo GeoJSON: 2024-11-30 a 2025-11-30
Intervalo CSV:     2025-12-01 a 2026-11-18
Dias em comum entre os arquivos: 0


,nulos,percentual (%)
DataHora,0,0.0000
Satelite,0,0.0000
Pais,0,0.0000
Estado,0,0.0000
Municipio,0,0.0000
Bioma,1,0.0001
DiaSemChuva,149263,9.1091
Precipitacao,149263,9.1091
RiscoFogo,149263,9.1091
FRP,0,0.0000


## 4. Classificação dos atributos

| Atributo | Tipo | Subtipo | Descrição |
| :--- | :--- | :--- | :--- |
| `Satelite` | Qualitativo | Nominal | Satélite de referência que detectou o foco (`NOAA-20`) |
| `Pais` | Qualitativo | Nominal | País do foco de queimada (`Brasil`) |
| `Estado` | Qualitativo | Nominal | Unidade federativa brasileira |
| `Municipio` | Qualitativo | Nominal | Município onde o foco foi registrado |
| `Bioma` | Qualitativo | Nominal | Bioma brasileiro (Amazônia, Cerrado, Caatinga, Mata Atlântica, Pantanal, Pampa) |
| `origem_formato` | Qualitativo | Nominal | Origem/formato do arquivo de coleta (`GeoJSON (2024-2025)` ou `CSV (2025-2026)`) |
| `ano_mes` *(derivado)* | Qualitativo | Ordinal | Período mensal ordenado cronologicamente (`YYYY-MM`) |
| `DiaSemChuva` | Quantitativo | Discreto | Número inteiro de dias consecutivos sem chuva antes do foco |
| `ano` *(derivado)* | Quantitativo | Discreto | Ano calendário do registro |
| `mes` *(derivado)* | Quantitativo | Discreto | Mês calendário do registro (`1` a `12`) |
| `DataHora` | Quantitativo | Contínuo | Instante temporal do foco (`datetime`) |
| `Precipitacao` | Quantitativo | Contínuo | Precipitação acumulada (mm) |
| `RiscoFogo` | Quantitativo | Contínuo | Índice de risco de fogo calculado pelo INPE (`0.0` a `1.0`) |
| `FRP` | Quantitativo | Contínuo | *Fire Radiative Power* — intensidade radiativa do foco (MW) |
| `Latitude` | Quantitativo | Contínuo | Coordenada geográfica de latitude (graus decimais) |
| `Longitude` | Quantitativo | Contínuo | Coordenada geográfica de longitude (graus decimais) |

*Nota:* A base bruta do INPE não possui atributos qualitativos ordinais nativos; o recorte `ano_mes` derivado atua como categoria temporal ordenada para agregações mensais.

## 5. Limpeza de `DataHora`, variáveis temporais e recorte 12×12 meses

In [4]:
# Converte para str antes do replace porque o concat mistura Timestamp (GeoJSON) e str (CSV)
datahora_str = df["DataHora"].astype(str)
qtd_bom = datahora_str.str.contains("\ufeff", regex=False).sum()

df["DataHora"] = pd.to_datetime(
    datahora_str.str.replace("\ufeff", "", regex=False),
    format="mixed",
)
assert df["DataHora"].isna().sum() == 0, "Foram gerados NaT na conversão de DataHora"

df["ano"] = df["DataHora"].dt.year
df["mes"] = df["DataHora"].dt.month
df["ano_mes"] = df["DataHora"].dt.strftime("%Y-%m")

print(f"Registros com BOM (\\ufeff) limpos: {qtd_bom:,}")
print(f"Dtype final de DataHora: {df['DataHora'].dtype}")
display(df[["DataHora", "ano", "mes", "ano_mes"]].head())

Registros com BOM (\ufeff) limpos: 8
Dtype final de DataHora: datetime64[us]


,DataHora,ano,mes,ano_mes
0,2024-11-30 03:46:00,2024,11,2024-11
1,2024-11-30 03:46:00,2024,11,2024-11
2,2024-11-30 03:46:00,2024,11,2024-11
3,2024-11-30 03:46:00,2024,11,2024-11
4,2024-11-30 03:46:00,2024,11,2024-11


In [5]:
mask_dia_isolado = df["DataHora"].dt.date == pd.Timestamp("2024-11-30").date()
qtd_removidos_dia = int(mask_dia_isolado.sum())
assert qtd_removidos_dia == 5_002, f"Esperado 5.002 focos em 2024-11-30, obtido {qtd_removidos_dia}"

df = df.loc[~mask_dia_isolado].copy()

print(f"Focos removidos de 2024-11-30: {qtd_removidos_dia:,}")
print(f"Total após recorte 12x12: {len(df):,}")
display(df.groupby("origem_formato")["ano_mes"].agg(["min", "max", "nunique"]))

Focos removidos de 2024-11-30: 5,002
Total após recorte 12x12: 1,633,620


,min,max,nunique
origem_formato,,,
CSV (2025-2026),2025-12,2026-11,12
GeoJSON (2024-2025),2024-12,2025-11,12


## 6. Padronização de `Estado` e remoção de duplicidades

In [6]:
df["Estado"] = df["Estado"].str.strip().str.upper()
print(f"Estados únicos ({df['Estado'].nunique()}): {sorted(df['Estado'].unique())}")

dup_exatas = int(df.duplicated().sum())
dup_espaco_temp = int(df.duplicated(subset=["DataHora", "Latitude", "Longitude"]).sum())

print(f"Duplicatas exatas: {dup_exatas:,}")
print(f"Contradições espaço-temporais (DataHora, Latitude, Longitude): {dup_espaco_temp:,}")

antes_dup = len(df)
df = df.drop_duplicates().drop_duplicates(subset=["DataHora", "Latitude", "Longitude"]).copy()
print(f"Total removido por duplicidade: {antes_dup - len(df):,}")
print(f"Total após deduplicação: {len(df):,}")

Estados únicos (27): ['ACRE', 'ALAGOAS', 'AMAPÁ', 'AMAZONAS', 'BAHIA', 'CEARÁ', 'DISTRITO FEDERAL', 'ESPÍRITO SANTO', 'GOIÁS', 'MARANHÃO', 'MATO GROSSO', 'MATO GROSSO DO SUL', 'MINAS GERAIS', 'PARANÁ', 'PARAÍBA', 'PARÁ', 'PERNAMBUCO', 'PIAUÍ', 'RIO DE JANEIRO', 'RIO GRANDE DO NORTE', 'RIO GRANDE DO SUL', 'RONDÔNIA', 'RORAIMA', 'SANTA CATARINA', 'SERGIPE', 'SÃO PAULO', 'TOCANTINS']


Duplicatas exatas: 0
Contradições espaço-temporais (DataHora, Latitude, Longitude): 4


Total removido por duplicidade: 4
Total após deduplicação: 1,633,616


## 7. Sentinelas `-999`, `Bioma` nulo e imputação com mediana

In [7]:
cols_sentinela = ["DiaSemChuva", "RiscoFogo"]
for col in cols_sentinela:
    qtd = int((df[col] == -999).sum())
    print(f"Valores -999 em {col}: {qtd:,}")

df[cols_sentinela] = df[cols_sentinela].replace(-999, np.nan)

bioma_nulos = int(df["Bioma"].isna().sum())
print(f"Registros com Bioma nulo: {bioma_nulos:,}")
df = df.dropna(subset=["Bioma"]).copy()
print(f"Total após remoção de Bioma nulo: {len(df):,}")

Valores -999 em DiaSemChuva: 21,274
Valores -999 em RiscoFogo: 13,628
Registros com Bioma nulo: 1


Total após remoção de Bioma nulo: 1,633,615


In [8]:
cols_imputar = ["DiaSemChuva", "Precipitacao", "RiscoFogo"]
print("Nulos antes da imputação:")
display(df[cols_imputar].isna().sum())

imputer = SimpleImputer(strategy="median")
df[cols_imputar] = imputer.fit_transform(df[cols_imputar])

medianas = dict(zip(cols_imputar, imputer.statistics_))
print("Medianas aplicadas:", medianas)
print("Nulos após a imputação:")
display(df[cols_imputar].isna().sum())

Nulos antes da imputação:


DiaSemChuva     170537
Precipitacao    149263
RiscoFogo       162891
dtype: int64

Medianas aplicadas: {'DiaSemChuva': np.float64(8.0), 'Precipitacao': np.float64(0.0), 'RiscoFogo': np.float64(1.0)}
Nulos após a imputação:


DiaSemChuva     0
Precipitacao    0
RiscoFogo       0
dtype: int64

## 8. Validação final e exportação em Parquet

In [9]:
df = df.reset_index(drop=True)

assert df.isna().sum().sum() == 0, "Ainda existem valores nulos na base final"
assert (df[["DiaSemChuva", "RiscoFogo"]] == -999).sum().sum() == 0, "Ainda existem sentinelas -999"

df.to_parquet(ARQUIVO_PARQUET, index=False)
tamanho_mb = ARQUIVO_PARQUET.stat().st_size / (1024 ** 2)

print(f"Base limpa salva em: {ARQUIVO_PARQUET} ({tamanho_mb:.2f} MB)")
print(f"Shape final: {df.shape}")
df.info()

Base limpa salva em: ..\dados\queimadas_limpo.parquet (34.78 MB)
Shape final: (1633615, 16)
<class 'pandas.DataFrame'>
RangeIndex: 1633615 entries, 0 to 1633614
Data columns (total 16 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   DataHora        1633615 non-null  datetime64[us]
 1   Satelite        1633615 non-null  str           
 2   Pais            1633615 non-null  str           
 3   Estado          1633615 non-null  str           
 4   Municipio       1633615 non-null  str           
 5   Bioma           1633615 non-null  str           
 6   DiaSemChuva     1633615 non-null  float64       
 7   Precipitacao    1633615 non-null  float64       
 8   RiscoFogo       1633615 non-null  float64       
 9   FRP             1633615 non-null  float64       
 10  Latitude        1633615 non-null  float64       
 11  Longitude       1633615 non-null  float64       
 12  origem_formato  1633615 non-null  str        